# 03 · Logistic regression

Predicts the **probability** of class 1 by squashing the linear predictor through the **sigmoid** σ(z) = 1 / (1 + e⁻ᶻ), optimizing **binary cross-entropy** (BCE) loss.

No closed form — fitting always runs an optimizer:

- **Gradient descent** (default): mini-batch, Nesterov momentum. Cheap steps, scales to large datasets.
- **Newton's method**: uses the Hessian (loss curvature) to converge in far fewer iterations, at a higher cost per step.

## The data

Binary target from log-odds `1.5·x1 − 2·x2 + 0.4`, turned into 0/1 labels with overlap (classes not perfectly separable). Target column holds integers `0`/`1`. Logistic regression is binary: it refuses any other target. For more than two classes use KNN, trees or boosting (their results are read with the multiclass metrics of [`06_metrics`](06_metrics.ipynb)).

In [1]:
import numpy as np
import polars as pl

from dolcestat.preprocessing import DolceSet
from dolcestat.linear_models import LogisticRegression

rng = np.random.default_rng(0)
n = 300
x1 = rng.normal(0, 1, n)
x2 = rng.normal(0, 1, n)
prob = 1 / (1 + np.exp(-(1.5 * x1 - 2.0 * x2 + 0.4)))
label = (rng.uniform(size=n) < prob).astype(int)

df = pl.DataFrame({"x1": x1, "x2": x2, "label": label})
data = DolceSet()
data.load_from_polars_dataframe(df, target_col="label")

## Fitting with gradient descent

`LogisticRegression` defaults to `optimizer="gradient descent"` (mini-batch, Nesterov momentum): each iteration nudges weights along the negative BCE gradient. Coefficients should land near the true 1.5, −2, 0.4.

In [2]:
model = LogisticRegression()        # optimizer="gradient descent"
model.fit(data)

print("weights [w1, w2, b]:", np.round(model.weights, 3))

weights [w1, w2, b]: [ 1.606 -1.878  0.211]


## Probabilities vs. classes

`model.predict(data)` returns a classification **analyzer**; `y_fit` holds predicted class-1 **probabilities** (sigmoid output). Threshold (default 0.5) to get hard labels.

In [3]:
report = model.predict(data)
probabilities = report.y_fit

predicted_class = (probabilities >= 0.5).astype(int)

print("first 6 probabilities:", np.round(probabilities[:6], 3))
print("first 6 predicted classes:", predicted_class[:6])

first 6 probabilities: [0.136 0.232 0.548 0.999 0.243 0.698]
first 6 predicted classes: [0 0 1 1 0 1]


## Choosing the optimizer

Accepts `"gradient descent"` (default) or `"newton"` — no `"closed form"` exists. Newton's method uses the Hessian to jump to the minimum of the local quadratic approximation: fewer iterations, more expensive steps. You can also pass a configured optimizer instance, as with linear regression. Details in [`04_optimization`](04_optimization.ipynb).

In [4]:
newton_model = LogisticRegression("newton").fit(data)

print("GD weights:    ", np.round(model.weights, 3))
print("Newton weights:", np.round(newton_model.weights, 3))

GD weights:     [ 1.606 -1.878  0.211]
Newton weights: [ 1.781 -2.183  0.222]


## Recap

You've fit a probabilistic classifier and turned probabilities into class predictions. Evaluating them (accuracy, precision, recall, ROC/AUC, threshold choice) is covered in [`06_metrics`](06_metrics.ipynb). Next: [`04_optimization`](04_optimization.ipynb).